<a href="https://colab.research.google.com/github/SofiiaBobr/goit_machine_learning/blob/main/hw7_nyc_taxi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ДЗ7 — Time-series ML features та JSONB у PostgreSQL
**Relational Databases for DS · Тема 7**

Джерело: NYC TLC Yellow Taxi, March 2024. За відсутності доступу до Parquet автоматично використовується детермінований stub (60 днів × 5 зон). Усі SQL-запити виконуються в PostgreSQL. Виводи EXPLAIN та Reflection створюються після Run All.

In [23]:
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

In [24]:
import os, re, time
from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown
pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)
def query(sql, params=None):
    with engine.connect() as c:
        return pd.read_sql_query(text(sql), c, params=params)
def run(sql):
    with engine.begin() as c:
        c.execute(text(sql))
def explain(sql):
    with engine.connect() as c:
        rows=c.execute(text('EXPLAIN (ANALYZE, BUFFERS) '+sql)).fetchall()
    lines=[r[0] for r in rows]
    print('\n'.join(lines))
    return '\n'.join(lines)
def duration(plan):
    m=re.search(r'Execution Time:\s*([0-9.]+) ms',plan)
    return float(m.group(1)) if m else None
print(query('SELECT version() AS postgres_version').iloc[0,0])

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 1. NYC Taxi: завантаження та daily aggregation
Для відтворюваності завантажуємо до 500 000 поїздок з офіційного Parquet. Якщо мережа недоступна, fallback чітко позначається в результатах і README.

In [25]:
URL='https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-03.parquet'
cols=['tpep_pickup_datetime','tpep_dropoff_datetime','PULocationID','DOLocationID','trip_distance','fare_amount','passenger_count']
source='NYC TLC Parquet'
try:
    path='/content/yellow_tripdata_2024-03.parquet' if os.path.isdir('/content') else '/tmp/yellow_tripdata_2024-03.parquet'
    if not os.path.exists(path): urlretrieve(URL,path)
    df=pd.read_parquet(path,columns=cols).head(500_000)
    df=df[(df.fare_amount>0)&(df.fare_amount<500)&(df.trip_distance>0)&(df.trip_distance<100)&df.PULocationID.notna()]
    df.to_sql('nyc_taxi_raw',engine,if_exists='replace',index=False,chunksize=10_000)
    run('DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE')
    run("""CREATE TABLE t7_hw_taxi_daily AS
    SELECT "PULocationID"::INTEGER AS pickup_zone_id,
      DATE_TRUNC('day',tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
      COUNT(*)::INTEGER AS trip_count,
      AVG(fare_amount)::NUMERIC(10,2) AS avg_fare,
      SUM(trip_distance)::NUMERIC(12,2) AS total_distance
    FROM nyc_taxi_raw GROUP BY 1,2""")
    run('ALTER TABLE t7_hw_taxi_daily ADD PRIMARY KEY (pickup_zone_id,ts_day)')
except Exception as e:
    source='DETERMINISTIC STUB (download unavailable)'
    print('Parquet unavailable:',str(e)[:300])
    run('DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE')
    run("""CREATE TABLE t7_hw_taxi_daily (
      pickup_zone_id INTEGER NOT NULL, ts_day DATE NOT NULL,
      trip_count INTEGER NOT NULL CHECK(trip_count>=0),
      avg_fare NUMERIC(10,2) NOT NULL CHECK(avg_fare>=0),
      total_distance NUMERIC(12,2) NOT NULL CHECK(total_distance>=0),
      PRIMARY KEY(pickup_zone_id,ts_day))""")
    run("""INSERT INTO t7_hw_taxi_daily
    SELECT z.zone_id, DATE '2024-03-31'-(59-n),
      500+n*7+z.zone_id*50+(n%5)*30,
      (12.5+z.zone_id*0.7+(n%4)*0.4)::NUMERIC(10,2),
      (1500+n*25+z.zone_id*120)::NUMERIC(12,2)
    FROM generate_series(0,59) n
    CROSS JOIN (VALUES(1),(2),(3),(4),(5)) z(zone_id)""")
print('SOURCE:',source)
display(query('SELECT COUNT(*) AS n_rows, COUNT(DISTINCT pickup_zone_id) AS zones, MIN(ts_day) AS first_day, MAX(ts_day) AS last_day FROM t7_hw_taxi_daily'))

SOURCE: NYC TLC Parquet


,n_rows,zones,first_day,last_day
0,1217,240,2024-02-29,2024-03-06


**Важливо:** LAG(7) і LAG(30) працюють із попередніми *рядками*, а не обов’язково календарними днями. Для повністю календарних lag-ознак у production потрібен щільний календар по кожній зоні. Тут використано шаблон завдання.

## 2. Time-series ML features — окремі групи

In [26]:
lag_sql="""SELECT pickup_zone_id,ts_day,trip_count,
 LAG(trip_count,1) OVER w AS trips_lag_1d,
 LAG(trip_count,7) OVER w AS trips_lag_7d,
 LAG(trip_count,30) OVER w AS trips_lag_30d
 FROM t7_hw_taxi_daily
 WINDOW w AS(PARTITION BY pickup_zone_id ORDER BY ts_day)
 ORDER BY pickup_zone_id,ts_day LIMIT 20"""
display(query(lag_sql))

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-01,1,NaN,None,None
1,1,2024-03-02,2,1.0,None,None
2,1,2024-03-03,2,2.0,None,None
3,1,2024-03-04,1,2.0,None,None
4,1,2024-03-05,1,1.0,None,None
5,3,2024-03-01,3,NaN,None,None
6,3,2024-03-02,1,3.0,None,None
7,3,2024-03-03,4,1.0,None,None
8,3,2024-03-04,2,4.0,None,None
9,3,2024-03-05,1,2.0,None,None


In [27]:
rolling7_sql="""SELECT pickup_zone_id,ts_day,avg_fare,
 AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
 COUNT(*) OVER w7 AS days_in_window
 FROM t7_hw_taxi_daily WINDOW w7 AS (
 PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
 RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW)
 ORDER BY pickup_zone_id,ts_day LIMIT 20"""
display(query(rolling7_sql))

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-01,105.00,105.000000,1
1,1,2024-03-02,95.00,100.000000,2
2,1,2024-03-03,51.00,83.666667,3
3,1,2024-03-04,40.00,72.750000,4
4,1,2024-03-05,120.00,82.200000,5
5,3,2024-03-01,39.17,39.170000,1
6,3,2024-03-02,21.50,30.335000,2
7,3,2024-03-03,25.03,28.566667,3
8,3,2024-03-04,39.15,31.212500,4
9,3,2024-03-05,45.50,34.070000,5


In [28]:
rolling30_sql="""SELECT pickup_zone_id,ts_day,avg_fare,
 STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
 COUNT(*) OVER w30 AS days_in_window
 FROM t7_hw_taxi_daily WINDOW w30 AS (
 PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
 RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW)
 ORDER BY pickup_zone_id,ts_day LIMIT 20"""
display(query(rolling30_sql))

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-01,105.00,NaN,1
1,1,2024-03-02,95.00,7.071068,2
2,1,2024-03-03,51.00,28.728615,3
3,1,2024-03-04,40.00,32.045541,4
4,1,2024-03-05,120.00,34.881227,5
5,3,2024-03-01,39.17,NaN,1
6,3,2024-03-02,21.50,12.494577,2
7,3,2024-03-03,25.03,9.350841,3
8,3,2024-03-04,39.15,9.289451,4
9,3,2024-03-05,45.50,10.273605,5


In [29]:
expanding_sql="""SELECT pickup_zone_id,ts_day,trip_count,
 SUM(trip_count) OVER we AS cumulative_trips,
 AVG(trip_count) OVER we AS expanding_avg_trips
 FROM t7_hw_taxi_daily WINDOW we AS (
 PARTITION BY pickup_zone_id ORDER BY ts_day
 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
 ORDER BY pickup_zone_id,ts_day LIMIT 20"""
display(query(expanding_sql))

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-01,1,1,1.000000
1,1,2024-03-02,2,3,1.500000
2,1,2024-03-03,2,5,1.666667
3,1,2024-03-04,1,6,1.500000
4,1,2024-03-05,1,7,1.400000
5,3,2024-03-01,3,3,3.000000
6,3,2024-03-02,1,4,2.000000
7,3,2024-03-03,4,8,2.666667
8,3,2024-03-04,2,10,2.500000
9,3,2024-03-05,1,11,2.200000


### Combined feature view

In [30]:
run('DROP VIEW IF EXISTS v_t7_hw_taxi_features CASCADE')
run("""CREATE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
 SELECT pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,
 LAG(trip_count,1) OVER w_pid AS trips_lag_1d,
 LAG(trip_count,7) OVER w_pid AS trips_lag_7d,
 LAG(trip_count,30) OVER w_pid AS trips_lag_30d,
 AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
 STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
 SUM(trip_count) OVER we AS cumulative_trips,
 AVG(trip_count) OVER we AS expanding_avg_trips
 FROM t7_hw_taxi_daily
 WINDOW w_pid AS(PARTITION BY pickup_zone_id ORDER BY ts_day),
 w7 AS(PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
 RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW),
 w30 AS(PARTITION BY pickup_zone_id ORDER BY ts_day::TIMESTAMP
 RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW),
 we AS(PARTITION BY pickup_zone_id ORDER BY ts_day
 ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW))
SELECT * FROM features_basic""")
display(query('SELECT * FROM v_t7_hw_taxi_features ORDER BY pickup_zone_id,ts_day LIMIT 20'))
display(query('SELECT COUNT(*) AS feature_rows FROM v_t7_hw_taxi_features'))

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,1,2024-03-01,1,105.00,0.04,NaN,None,None,105.000000,NaN,1,1.000000
1,1,2024-03-02,2,95.00,18.43,1.0,None,None,100.000000,7.071068,3,1.500000
2,1,2024-03-03,2,51.00,0.09,2.0,None,None,83.666667,28.728615,5,1.666667
3,1,2024-03-04,1,40.00,0.01,2.0,None,None,72.750000,32.045541,6,1.500000
4,1,2024-03-05,1,120.00,0.03,1.0,None,None,82.200000,34.881227,7,1.400000
5,3,2024-03-01,3,39.17,34.00,NaN,None,None,39.170000,NaN,3,3.000000
6,3,2024-03-02,1,21.50,1.80,3.0,None,None,30.335000,12.494577,4,2.000000
7,3,2024-03-03,4,25.03,24.13,1.0,None,None,28.566667,9.350841,8,2.666667
8,3,2024-03-04,2,39.15,16.86,4.0,None,None,31.212500,9.289451,10,2.500000
9,3,2024-03-05,1,45.50,13.90,2.0,None,None,34.070000,10.273605,11,2.200000


,feature_rows
0,1217


## 3. JSONB event log — 50 000 synthetic events, 5 event types

In [31]:
run('DROP TABLE IF EXISTS t7_hw_events CASCADE')
run("""CREATE TABLE t7_hw_events (
 event_id INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
 user_id INTEGER NOT NULL,event_ts TIMESTAMPTZ NOT NULL DEFAULT NOW(),
 event_type TEXT NOT NULL,payload JSONB NOT NULL)""")
run("""INSERT INTO t7_hw_events(user_id,event_ts,event_type,payload)
SELECT (n%1000)+1,
 TIMESTAMPTZ '2024-04-01 00:00:00+00'-((n%60)||' days')::INTERVAL,
 CASE n%5 WHEN 0 THEN 'page_view' WHEN 1 THEN 'click'
 WHEN 2 THEN 'add_to_cart' WHEN 3 THEN 'checkout' ELSE 'recommendation_shown' END,
 CASE n%5
 WHEN 0 THEN jsonb_build_object('url','/product/'||((n%200)+1),
 'referrer',(ARRAY['google','direct','email','fb'])[(n%4)+1],
 'device',(ARRAY['mobile','desktop','tablet'])[(n%3)+1])
 WHEN 1 THEN jsonb_build_object('element_id','btn_'||((n%50)+1),
 'page_url','/category/'||((n%20)+1),
 'device',(ARRAY['mobile','desktop'])[(n%2)+1])
 WHEN 2 THEN jsonb_build_object('product_id',(n%200)+1,
 'price',((n%100)+10)::NUMERIC(10,2),'quantity',(n%5)+1,
 'tags',jsonb_build_array((ARRAY['promo','new','clearance'])[(n%3)+1]))
 WHEN 3 THEN jsonb_build_object('order_id','o_'||(n+100000),
 'total',((n%500)+50)::NUMERIC(10,2),'items_count',(n%10)+1,
 'payment_method',(ARRAY['card','paypal','crypto'])[(n%3)+1])
 ELSE jsonb_build_object('model_name','rec_v'||((n%5)+1),
 'model_version','1.'||(n%10),
 'recommendations',jsonb_build_array((n%200)+1,((n+1)%200)+1,((n+2)%200)+1),
 'experiment','exp_'||((n%3)+1)) END
FROM generate_series(1,50000) n""")
run('ANALYZE t7_hw_events')
display(query('SELECT event_type,COUNT(*) AS n_events FROM t7_hw_events GROUP BY event_type ORDER BY event_type'))

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### JSONB operators: `@>`, `?`, `?|`, JSONPath

In [32]:
display(query("SELECT event_id,user_id,event_ts,payload FROM t7_hw_events WHERE event_type='checkout' AND payload @> '{\"payment_method\":\"card\"}'::jsonb LIMIT 10"))
display(query("SELECT event_id,event_type,payload FROM t7_hw_events WHERE payload ? 'tags' LIMIT 10"))
display(query("SELECT event_id,event_type,payload FROM t7_hw_events WHERE payload ?| ARRAY['experiment','model_version'] LIMIT 10"))

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


In [33]:
display(query("SELECT event_id,jsonb_path_query(payload,'$.recommendations[*]') AS recommended_pid FROM t7_hw_events WHERE event_type='recommendation_shown' LIMIT 15"))
display(query("SELECT event_id,payload->>'order_id' AS order_id,(payload->>'total')::NUMERIC(10,2) AS total FROM t7_hw_events WHERE event_type='checkout' AND jsonb_path_exists(payload,'$.total ? (@ > 300)') LIMIT 10"))

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


## 4. GIN vs expression B-tree — реальні EXPLAIN (ANALYZE, BUFFERS)

In [34]:
for name in ['idx_t7_events_payload_gin_pathops','idx_t7_events_payload_gin_ops','idx_t7_events_device_btree']:
    run(f'DROP INDEX IF EXISTS {name}')
run('ANALYZE t7_hw_events')
contain_sql="SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('payment_method','card')"
print('BEFORE GIN')
plan_contain_before=explain(contain_sql)

BEFORE GIN
Aggregate  (cost=1758.18..1758.19 rows=1 width=8) (actual time=59.497..59.507 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3272 width=0) (actual time=0.013..59.090 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.136 ms
Execution Time: 59.530 ms


In [35]:
run('CREATE INDEX idx_t7_events_payload_gin_pathops ON t7_hw_events USING GIN(payload jsonb_path_ops)')
run('ANALYZE t7_hw_events')
display(query("SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS gin_index_size"))
print('AFTER GIN')
plan_contain_after=explain(contain_sql)
print('OTHER CONTAINMENT')
plan_referrer=explain("SELECT COUNT(*) FROM t7_hw_events WHERE payload @> jsonb_build_object('referrer','google')")

,gin_index_size
0,1144 kB


AFTER GIN
Aggregate  (cost=1098.96..1098.97 rows=1 width=8) (actual time=5.127..5.128 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=35.88..1089.95 rows=3605 width=0) (actual time=0.821..4.825 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..34.97 rows=3605 width=0) (actual time=0.366..0.366 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=14
Planning Time: 0.231 ms
Execution Time: 5.158 ms
OTHER CONTAINMENT
Aggregate  (cost=1080.90..1080.91 rows=1 width=8) (actual time=2.880..2.880 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=31.71..1073.87 rows=2811 width=0) (actual time=0.372..2.764 rows

In [36]:
device_sql="SELECT COUNT(*) FROM t7_hw_events WHERE payload ->> 'device' = 'mobile'"
print('BEFORE B-TREE')
plan_device_before=explain(device_sql)

BEFORE B-TREE
Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=14.489..14.490 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.017..13.906 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.073 ms
Execution Time: 14.518 ms


In [37]:
run("CREATE INDEX idx_t7_events_device_btree ON t7_hw_events ((payload ->> 'device'))")
run('ANALYZE t7_hw_events')
print('AFTER B-TREE')
plan_device_after=explain(device_sql)

AFTER B-TREE
Aggregate  (cost=1241.94..1241.95 rows=1 width=8) (actual time=2.515..2.516 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.64..1221.18 rows=8303 width=0) (actual time=0.517..2.141 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.56 rows=8303 width=0) (actual time=0.395..0.395 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=5 read=1
Planning Time: 0.224 ms
Execution Time: 2.543 ms


## 5. JSONB vs normalized checkout

In [38]:
run('DROP TABLE IF EXISTS t7_hw_checkouts_norm CASCADE')
run("""CREATE TABLE t7_hw_checkouts_norm (
 event_id INTEGER PRIMARY KEY,user_id INTEGER NOT NULL,
 event_ts TIMESTAMPTZ NOT NULL,order_id TEXT NOT NULL,
 total NUMERIC(10,2) NOT NULL,
 items_count INTEGER NOT NULL CHECK(items_count>0),
 payment_method TEXT NOT NULL CHECK(payment_method IN ('card','paypal','crypto')))""")
run("""INSERT INTO t7_hw_checkouts_norm
SELECT event_id,user_id,event_ts,payload->>'order_id',
 (payload->>'total')::NUMERIC(10,2),
 (payload->>'items_count')::INTEGER,payload->>'payment_method'
FROM t7_hw_events WHERE event_type='checkout'""")
run('CREATE INDEX idx_t7_checkouts_norm_pm ON t7_hw_checkouts_norm(payment_method)')
run('ANALYZE t7_hw_checkouts_norm')
display(query('SELECT COUNT(*) AS n_checkouts FROM t7_hw_checkouts_norm'))

,n_checkouts
0,10000


In [39]:
json_agg_sql="""SELECT payload->>'payment_method' AS payment_method,
 AVG((payload->>'total')::NUMERIC) AS avg_total,COUNT(*) AS n
 FROM t7_hw_events WHERE event_type='checkout'
 GROUP BY payload->>'payment_method' ORDER BY avg_total DESC LIMIT 3"""
norm_agg_sql="""SELECT payment_method,AVG(total) AS avg_total,COUNT(*) AS n
 FROM t7_hw_checkouts_norm GROUP BY payment_method ORDER BY avg_total DESC LIMIT 3"""
print('JSONB QUERY')
display(query(json_agg_sql)); plan_json_agg=explain(json_agg_sql)
print('NORMALIZED QUERY')
display(query(norm_agg_sql)); plan_norm_agg=explain(norm_agg_sql)

JSONB QUERY


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


Limit  (cost=1954.11..1954.11 rows=3 width=72) (actual time=13.216..13.217 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1954.11..1967.94 rows=5533 width=72) (actual time=13.215..13.216 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.60..1882.59 rows=5533 width=72) (actual time=13.195..13.207 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.94 rows=9977 width=132) (actual time=0.012..8.185 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning Time: 0.115 ms
Execution Time: 13.264 ms
NORMALIZED QUERY


,payment_method,avg_total,n
0,paypal,300.524752,3333
1,card,300.500000,3334
2,crypto,300.475248,3333


Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=4.031..4.033 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=4.031..4.032 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=4.017..4.019 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.008..0.890 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning Time: 0.072 ms
Execution Time: 4.057 ms


### Bonus: MongoDB-equivalent (pseudocode, не виконується в PostgreSQL)
```javascript
db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {_id: "$payload.payment_method", avg_total: {$avg: {$toDouble: "$payload.total"}}, n: {$sum: 1}}},
  {$sort: {avg_total: -1}}, {$limit: 3}
]);
```
PostgreSQL JSONB зручний для SQL/JOIN/транзакцій, MongoDB — для document-first workflows. Масовий clickstream можна зберігати в data lake, а curated ML features — у typed SQL-таблицях.

## Reflection (250–300 words)
Текст нижче формується на основі **реальних** EXPLAIN-output після запуску. Це важливо: результати залежать від середовища та планувальника PostgreSQL.

In [44]:
def operator(plan):
    names=['Bitmap Index Scan','Bitmap Heap Scan','Index Only Scan','Index Scan','Seq Scan','Parallel Seq Scan','HashAggregate','Hash Aggregate','GroupAggregate','Sort']
    return ', '.join(x for x in names if x in plan) or 'other plan operators'
def cost(plan):
    m=re.search(r'cost=([0-9.]+)\.\.([0-9.]+)',plan)
    return m.group(2) if m else 'N/A'
metrics=[('contain_before',plan_contain_before),('contain_after',plan_contain_after),('device_before',plan_device_before),('device_after',plan_device_after),('json_agg',plan_json_agg),('norm_agg',plan_norm_agg)]
display(pd.DataFrame([{'scenario':k,'execution_ms':duration(v),'estimated_total_cost':cost(v),'operators':operator(v)} for k,v in metrics]))
reflection=f"""In this experiment, I compared PostgreSQL JSONB queries before and after creating indexes, using actual EXPLAIN (ANALYZE, BUFFERS) outputs rather than assuming an improvement. For the containment predicate, the estimated total cost changed from {cost(plan_contain_before)} to {cost(plan_contain_after)}, while measured execution time changed from {duration(plan_contain_before)} ms to {duration(plan_contain_after)} ms. Before indexing, the plan included {operator(plan_contain_before)}; after adding the GIN jsonb_path_ops index, it included {operator(plan_contain_after)}. A bitmap scan is not guaranteed: PostgreSQL can prefer a sequential scan when selectivity is low or a table is small.

GIN is designed to index JSONB containment and compatible JSONPath searches, but it is not a universal solution for expressions such as payload ->> 'device' = 'mobile'. For this text-equality predicate, an expression B-tree index matches the extracted scalar value directly. The measured times were {duration(plan_device_before)} ms before and {duration(plan_device_after)} ms after that index, with plans containing {operator(plan_device_before)} and {operator(plan_device_after)}, respectively. Indexes accelerate selective reads, but consume disk space and add maintenance work to inserts, updates, and deletes.

For the checkout aggregation, JSONB took {duration(plan_json_agg)} ms, compared with {duration(plan_norm_agg)} ms for the normalized table. I would keep heterogeneous, evolving event payloads in JSONB, especially for experimentation and ingestion, while using normalized typed tables for stable payment, order, and ML training fields requiring constraints and predictable analytics. The correct choice depends on observed workload, not blanket claim that normalization is always faster.

To introduce a refund event in JSONB, I would add a new event_type and document and validate its payload schema in ingestion code; the base table need not change. In a normalized design, I would create a refunds table with typed refund_id, amount, reason and foreign keys, plus migrations, tests and indexes. This favors integrity but costs schema-management effort."""
words=len(reflection.split()); print('Reflection word count:',words)
display(Markdown(reflection))

,scenario,execution_ms,estimated_total_cost,operators
0,contain_before,59.530,1758.19,Seq Scan
1,contain_after,5.158,1098.97,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
2,device_before,14.518,1750.63,Seq Scan
3,device_after,2.543,1241.95,"Bitmap Index Scan, Bitmap Heap Scan, Index Scan"
4,json_agg,13.264,1954.11,"Seq Scan, HashAggregate, Sort"
5,norm_agg,4.057,269.07,"Seq Scan, HashAggregate, Sort"


Reflection word count: 300


In this experiment, I compared PostgreSQL JSONB queries before and after creating indexes, using actual EXPLAIN (ANALYZE, BUFFERS) outputs rather than assuming an improvement. For the containment predicate, the estimated total cost changed from 1758.19 to 1098.97, while measured execution time changed from 59.53 ms to 5.158 ms. Before indexing, the plan included Seq Scan; after adding the GIN jsonb_path_ops index, it included Bitmap Index Scan, Bitmap Heap Scan, Index Scan. A bitmap scan is not guaranteed: PostgreSQL can prefer a sequential scan when selectivity is low or a table is small.

GIN is designed to index JSONB containment and compatible JSONPath searches, but it is not a universal solution for expressions such as payload ->> 'device' = 'mobile'. For this text-equality predicate, an expression B-tree index matches the extracted scalar value directly. The measured times were 14.518 ms before and 2.543 ms after that index, with plans containing Seq Scan and Bitmap Index Scan, Bitmap Heap Scan, Index Scan, respectively. Indexes accelerate selective reads, but consume disk space and add maintenance work to inserts, updates, and deletes.

For the checkout aggregation, JSONB took 13.264 ms, compared with 4.057 ms for the normalized table. I would keep heterogeneous, evolving event payloads in JSONB, especially for experimentation and ingestion, while using normalized typed tables for stable payment, order, and ML training fields requiring constraints and predictable analytics. The correct choice depends on observed workload, not blanket claim that normalization is always faster.

To introduce a refund event in JSONB, I would add a new event_type and document and validate its payload schema in ingestion code; the base table need not change. In a normalized design, I would create a refunds table with typed refund_id, amount, reason and foreign keys, plus migrations, tests and indexes. This favors integrity but costs schema-management effort.

In [45]:
checks=query("""SELECT (SELECT COUNT(*) FROM t7_hw_taxi_daily) AS taxi_rows,
 (SELECT COUNT(*) FROM v_t7_hw_taxi_features) AS feature_rows,
 (SELECT COUNT(*) FROM t7_hw_events) AS events,
 (SELECT COUNT(DISTINCT event_type) FROM t7_hw_events) AS event_types,
 (SELECT COUNT(*) FROM t7_hw_checkouts_norm) AS checkouts""")
display(checks)
r=checks.iloc[0]
assert r.taxi_rows>100 and r.feature_rows>0 and r.events==50000 and r.event_types>=5 and r.checkouts>0
assert all(duration(v) is not None for _,v in metrics)
assert 250<=words<=300, f'Reflection has {words} words; expected 250–300'
print('FINAL CHECK PASSED')

,taxi_rows,feature_rows,events,event_types,checkouts
0,1217,1217,50000,5,10000


FINAL CHECK PASSED
